In [1]:
import os
from dotenv import load_dotenv
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType
from google.cloud import bigquery

In [2]:
gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

True

In [3]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Transform Stage') \
        .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .getOrCreate() 

26/03/29 16:29:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [41]:
PostSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

In [5]:
bq_client = bigquery.Client()

project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
table_id = f"{project_id}:{dataset}.locations"

df_locations = spark.read.format("bigquery") \
                    .option('table', table_id) \
                    .load()

In [65]:
df_locations.count()

2791

In [6]:
df_locations.show(10)

+--------+--------------------+---------+----------+-------------+
|    City|            Location| Latitude| Longitude|High_Accuracy|
+--------+--------------------+---------+----------+-------------+
|Antipolo|MARCOS HIGHWAY MA...|14.622408|121.110111|          1.0|
|Antipolo|MARCOS HIGHWAY MA...|14.624459|121.119655|          1.0|
|Antipolo|C5 KATIPUNAN AVE ...|14.629705|121.074095|          1.0|
|Antipolo|C5 KATIPUNAN AVE ...|14.651395|121.074415|          1.0|
|  Cainta|          C5 MERCURY|14.607252|121.078585|          0.0|
|  Cainta|MARCOS HIGHWAY SA...|14.621608|121.106145|          1.0|
|  Cainta|MARCOS HIGHWAY TO...|14.621658|121.106498|          1.0|
|  Cainta| MARCOS HIGHWAY PLDT|14.621872|121.107543|          1.0|
|  Cainta|C5 KATIPUNAN MAYN...|14.655984|121.074458|          1.0|
|  Cainta|C5 KATIPUNAN MAGS...|14.657564|121.074422|          1.0|
+--------+--------------------+---------+----------+-------------+
only showing top 10 rows



In [7]:
from google.cloud import storage
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

BUCKET_NAME = os.getenv('BUCKET_NAME')
SCRAPE_FOLDER="raw/scrape"

gcs_client = storage.Client()
bucket = gcs_client.bucket(BUCKET_NAME)

# blobs = list(bucket.list_blobs(prefix=SCRAPE_FOLDER))
# files = [blob for blob in blobs if not blob.name.endswith('/')] 

In [8]:
PHT = ZoneInfo("Asia/Manila")
now = datetime.now(PHT)
yesterday = now - timedelta(days=1)

year = yesterday.strftime("%Y")
month = yesterday.strftime("%m")
day = yesterday.strftime("%d")

filename = f"{SCRAPE_FOLDER}/{year}/{month}/scrape_data_{year}{month}{day}.csv"
filename

'raw/scrape/2026/03/scrape_data_20260329.csv'

In [9]:
stats = storage.Blob(bucket=bucket, name=filename).exists()
stats

True

In [10]:
RawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])

In [11]:
def gcs_file_read(bucket_name, filepath):
    df = spark.read \
        .option("header", True) \
        .option("multiline", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("ignoreLeadingWhiteSpace", True) \
        .option("ignoreTrailingWhiteSpace", True) \
        .schema(RawSchema) \
        .csv(f"gs://{bucket_name}/{filepath}")
    
    return df

In [12]:
if stats:
    df_raw = gcs_file_read(BUCKET_NAME, filename)
    

In [17]:
df_raw.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+----------+
|content                                                                                                                                                                         |tweetlinkid                                    |created_at|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+----------+
|MMDA ALERT: Road crash incident at Bonny Serrano Libis EB involving trailer truck as of 9:30 PM. One lane occupied. MMDA enforcers are on site managing traffic. #mmda          |https://x.com/MMDA/status/2038249984932687992#m|2026-03-29|
|MMDA ALERT: Stalled dump truck due to clutch tr

In [14]:
PartialPostSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

In [ ]:
from utils.PostParser import post_parser

parse_all_udf = F.udf(post_parser, PartialPostSchema)

In [16]:
df_raw.printSchema()

root
 |-- content: string (nullable = true)
 |-- tweetlinkid: string (nullable = true)
 |-- created_at: date (nullable = true)



In [19]:
df_temp = df_raw.withColumn("parsed", 
                            parse_all_udf(
                                F.upper(df_raw['content']),
                                df_raw['created_at'],
                                df_raw['tweetlinkid']
                                )
                            )

In [20]:
df_temp.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content                                                                                                                                                                         |tweetlinkid                                    |created_at|parsed                                                                                                                                                                                 

In [21]:
df_test = df_temp.select(
    F.col("parsed.Date").alias("Date"),
    F.col("parsed.Time").alias("Time"),
    F.col("parsed.Location").alias("Location"),
    F.col("parsed.Direction").alias("Direction"),
    F.col("parsed.Type").alias("Type"),
    F.col("parsed.Lanes_Blocked").alias("Lanes_Blocked"),
    F.col("parsed.Involved").alias("Participants"),
    F.col("parsed.Tweet").alias("Tweet"),
    F.col("parsed.Source").alias("Source")
)

In [22]:
df_test.show(5, truncate=False)

+----------+-------+-------------------------------+---------+----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Date      |Time   |Location                       |Direction|Type                                    |Lanes_Blocked|Participants      |Tweet                                                                                                                                                                           |Source                                         |
+----------+-------+-------------------------------+---------+----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
enriched_df3 = df_locations.join(df_test, on="Location", how='right')
enriched_df3.show(5, truncate=False)

+-------------------------------+-----------+---------+----------+-------------+----------+-------+---------+----------------------------------------+-------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Location                       |City       |Latitude |Longitude |High_Accuracy|Date      |Time   |Direction|Type                                    |Lanes_Blocked|Participants      |Tweet                                                                                                                                                                           |Source                                         |
+-------------------------------+-----------+---------+----------+-------------+----------+-------+---------+----------------------------------------+-------------+------------------

In [28]:
missing_df = enriched_df3.filter(F.col('City').isNull())

In [30]:
missing_df.count()

7

In [33]:
unique_df = missing_df.select("Location").distinct()
unique_df.show()

+--------------------+
|            Location|
+--------------------+
|CONGRESSIONAL AVE...|
| BONNY SERRANO LIBIS|
|C5 J. VARGAS AVE....|
|QUEZON AVE., DEFE...|
|EDSA PASAY TAFT F...|
|EDSA P. TUAZON TU...|
|COMMONWEALTH IN F...|
+--------------------+



In [34]:
unique_missing = [row.Location for row in unique_df.collect()]
unique_missing

['CONGRESSIONAL AVE. TANDANG SORA',
 'BONNY SERRANO LIBIS',
 'C5 J. VARGAS AVE. BEFORE MERALCO FLYOVER',
 'QUEZON AVE., DEFENSOR SANTIAGO AVE. AFTER INTERSECTION',
 'EDSA PASAY TAFT FRONT OF DON ALDRIN',
 'EDSA P. TUAZON TUNNEL',
 'COMMONWEALTH IN FRONT OF SHOPWISE']

In [31]:
missing_df.show()

+--------------------+----+--------+---------+-------------+----------+--------+---------+--------------------+-------------+------------------+--------------------+--------------------+
|            Location|City|Latitude|Longitude|High_Accuracy|      Date|    Time|Direction|                Type|Lanes_Blocked|      Participants|               Tweet|              Source|
+--------------------+----+--------+---------+-------------+----------+--------+---------+--------------------+-------------+------------------+--------------------+--------------------+
| BONNY SERRANO LIBIS|null|    null|     null|         null|2026-03-29| 9:30 PM|       EB| ROAD CRASH INCIDENT|            1|     TRAILER TRUCK|MMDA ALERT: ROAD ...|https://x.com/MMD...|
|CONGRESSIONAL AVE...|null|    null|     null|         null|2026-03-29| 6:22 PM|       WB|STALLED DUMP TRUC...|            1|        DUMP TRUCK|MMDA ALERT: STALL...|https://x.com/MMD...|
|EDSA PASAY TAFT F...|null|    null|     null|         null|2026-

In [35]:
LocationDetailSchema = StructType([
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True)
])

In [59]:
import requests
import re
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv('AZURE_KEY')
CLIENT_ID = os.getenv('CLIENT_ID')
API_VERSION = os.getenv('API_VERSION')

accuracy_map = {
    "High": 1,
    "Medium": 0.5,
    "Low": 0
}

def create_batch_items(locations):

    batch_items = []

    for loc in locations:
        batch_items.append({
            "addressLine": loc,
            "adminDistrict": "Metro Manila",
            "countryRegion": "PH",
            "top": 1
        })

    return {"batchItems": batch_items}

def resolve_locations(spark, locations):
    body = create_batch_items(locations)

    url = 'https://atlas.microsoft.com/geocode:batch'

    params = {
        "api-version": "2026-01-01",
    }
    
    headers = {
        "Accept-Language": "en-US",
        "x-ms-client-id": CLIENT_ID,
        "subscription-key": KEY
    }

    response = requests.post(url=url, params=params, headers=headers, json=body)
    data = response.json()
    items = data['batchItems']

    details = []
    for item in items:
        feature = item.get('features', [None])[0] if item.get('features') else None

        if feature:
            geometry = feature.get('geometry', None)
            coordinates = geometry.get('coordinates', [None, None])
            longitude = float(coordinates[0])
            latitude = float(coordinates[1])

            properties = feature.get('properties', None)
            address = properties.get('address', None)
            city = address.get('locality', None)

            confidence = properties.get('confidence', None)
            accuracy = float(accuracy_map.get(confidence, 0))
        else:
            longitude = latitude = city = None
            accuracy = 0.0
        
        if city in ["Pasay", "Pasig", "Makati"]:
            city = city.strip() + " City"
        
        if city == "Kalookan City":
            city = "Caloocan City"

        pattern = re.compile(r'Para.*aque')
        if re.fullmatch(pattern, city):
            city = "Paranaque"
        
        details.append({
            "Longitude": longitude,
            "Latitude": latitude,
            "City": city,
            "High_Accuracy": accuracy
        })
    
    output_details = list(zip(locations, details))
    flattened_data = [(detail["City"], loc, detail["Latitude"], detail["Longitude"], detail["High_Accuracy"]) for loc, detail in output_details]

    # Create DataFrame
    df = spark.createDataFrame(flattened_data, LocationDetailSchema)
    return df

In [60]:
resolved_details = resolve_locations(spark, unique_missing)

In [61]:
resolved_details.show(truncate=False)

+-----------+------------------------------------------------------+----------------+----------------+-------------+
|City       |Location                                              |Latitude        |Longitude       |High_Accuracy|
+-----------+------------------------------------------------------+----------------+----------------+-------------+
|Quezon City|CONGRESSIONAL AVE. TANDANG SORA                       |14.6720084500126|121.043412799994|0.5          |
|Quezon City|BONNY SERRANO LIBIS                                   |14.6118801501389|121.059462449932|0.5          |
|Pasig City |C5 J. VARGAS AVE. BEFORE MERALCO FLYOVER              |14.5788869500001|121.063756549994|0.0          |
|Quezon City|QUEZON AVE., DEFENSOR SANTIAGO AVE. AFTER INTERSECTION|14.6491651000015|121.03949610001 |0.0          |
|Pasay City |EDSA PASAY TAFT FRONT OF DON ALDRIN                   |14.5372725859068|120.994162993509|0.0          |
|Quezon City|EDSA P. TUAZON TUNNEL                              

In [64]:
spark.conf.set('temporaryGcsBucket', 'tempresolvedlocation')

resolved_details.write \
                .format("bigquery") \
                .option('table', table_id) \
                .mode("append") \
                .save()
                

In [67]:
def load_locations_df(spark, table_id):
    return spark.read.format("bigquery") \
                .option('table', table_id) \
                .load()

df_locations = load_locations_df(spark, table_id)

In [69]:
def get_locations_from_bq(df_locations, raw_locations):
    return df_locations.join(raw_locations, on="Location", how='right')

In [75]:
df_locations.printSchema()

root
 |-- City: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- High_Accuracy: double (nullable = true)



In [76]:
enriched_df3.show()

26/03/29 19:30:30 ERROR Executor: Exception in task 0.0 in stage 48.0 (TID 53)1]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/app/src/02_transform/utils/PostParser.py", line 5, in <module>
    from utils.LocationFunctions import query_location, update_locations_bq, get_geocode
ImportError: cannot import name 'query_location' from 'utils.LocationFunctions' (/app/src/02_transform/utils/LocationFunctions.py)

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:561)
	at org.apache.spark.sql.execution.python.PythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.PythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:514)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:4

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/app/src/02_transform/utils/PostParser.py", line 5, in <module>
    from utils.LocationFunctions import query_location, update_locations_bq, get_geocode
ImportError: cannot import name 'query_location' from 'utils.LocationFunctions' (/app/src/02_transform/utils/LocationFunctions.py)


In [ ]:
df_full_parsed = df_locations.join(raw_locations, on="Location", how='right')

In [71]:
def get_missing_locations(enriched_df):
    # Get null values in City
    missing_df = enriched_df.filter(F.col('City').isNull())

    # Get unique values 
    missing_locations_df = missing_df.select("location").distinct()

    # Convert to array
    missing_locations = [row.location for row in missing_locations_df.collect()]

    return missing_locations